In [1]:
import pandas as pd

In [2]:
columns = ['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
           'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
           'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
           'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
           'tunnel_parents', 'ip_proto']

In [3]:
df = pd.read_csv("../data/raw/zeek-captures/conn.log",sep='\t',comment='#', names = columns)

In [4]:
print(df.shape)
df.head()

(2540, 22)


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
0,1.790911e+09,C946Oi3wl6FPOzvrNb,192.168.150.129,41656,45.33.32.156,80,tcp,-,-,-,...,T,F,0,C,0,0,0,0,-,6
1,1.790911e+09,CLF8yz0b7W9O25B6e,192.168.150.129,41668,45.33.32.156,80,tcp,-,-,-,...,T,F,0,C,0,0,0,0,-,6
2,1.790911e+09,CJQ4HJ3oYwaRfQSIH6,192.168.150.129,40080,45.33.32.156,993,tcp,-,-,-,...,T,F,0,C,0,0,0,0,-,6
3,1.790911e+09,CIVhFN21dHJQcwI0H8,192.168.150.129,58012,45.33.32.156,110,tcp,-,-,-,...,T,F,0,C,0,0,0,0,-,6
4,1.790911e+09,ChMrInJuilfpmOjAb,192.168.150.129,45786,45.33.32.156,111,tcp,-,-,-,...,T,F,0,C,0,0,0,0,-,6


In [5]:
result = (
    df.groupby("id.orig_h")["id.resp_p"]
      .nunique()
      .reset_index(name="unique_ports")
)

In [6]:
result['is_scanner'] = result['unique_ports'] > 50

In [7]:
print(result)

         id.orig_h  unique_ports  is_scanner
0    192.168.150.1             1       False
1  192.168.150.129          1000        True
2     45.33.32.156           486        True


## Limitation: the detector also flags the scan *target*

The detector flags any host that contacted more than 50 distinct destination
ports. On this capture it correctly flags the scanner
(`192.168.150.129`, 1000 ports) — but it also flags `45.33.32.156`
(scanme.nmap.org, 486 ports), which was the **target** of the scan, not an
attacker.

**Why this happens:** the detector counts distinct destination ports per
source host, but a scan produces bidirectional traffic. The target responds
from many ports, so Zeek logs the target as a source associated with many
ports too. Distinct-port-count alone can't distinguish the host that
*initiated* the connections (the scanner) from the host that *responded*
(the victim).

**How it could be refined:** use connection direction to tell initiator from
responder — for example, the scanner is consistently the originator
(`id.orig_h`) of large numbers of failed or reset connections
(`conn_state` values like `RSTRH`/`REJ`/`S0`), while the target's traffic is
responses. Filtering to failed, outbound-initiated connections before counting
ports would isolate the true scanner.

In [8]:
df['conn_state'].value_counts()

conn_state
RSTRH    1501
OTH      1032
SHR         3
RSTO        3
S0          1
Name: count, dtype: int64

In [9]:
df.groupby('id.orig_h')['conn_state'].value_counts()

id.orig_h        conn_state
192.168.150.1    S0               1
192.168.150.129  RSTRH         1501
                 OTH            540
                 RSTO             3
                 SHR              3
45.33.32.156     OTH            492
Name: count, dtype: int64

In [10]:
filtered_df = df[df['conn_state'] == 'RSTRH']

In [11]:
result2 = (
    filtered_df.groupby("id.orig_h")["id.resp_p"]
      .nunique()
      .reset_index(name="unique_ports")
)

In [12]:
print(result2)

         id.orig_h  unique_ports
0  192.168.150.129           793


In [13]:
result2['is_scanner'] = result2['unique_ports'] > 50

In [14]:
print(result2)

         id.orig_h  unique_ports  is_scanner
0  192.168.150.129           793        True


## Refinement: distinguishing the scanner from the scan target

The first detector flagged **both** the scanner (`192.168.150.129`) and the
scan *target* (`45.33.32.156`), because it counted distinct destination ports
across all connections — and a scan's bidirectional traffic associates the
target with many ports too.

Examining `conn_state` by host revealed the discriminator:

| Host | Traffic pattern |
|------|-----------------|
| `192.168.150.129` (scanner) | 1501 `RSTRH` — probes reset by the responder |
| `45.33.32.156` (target)     | 492 `OTH`, **zero `RSTRH`** |

The scanner's probes are overwhelmingly `RSTRH` (connection reset by the host
being contacted), while the target's return traffic has none. Filtering to
`RSTRH` connections before counting ports isolates the true scanner: the
target drops out entirely, leaving only `192.168.150.129` flagged (793 ports
that actively refused the probe — an even cleaner scan signal than the raw
port count).

**Note:** the exact connection states indicating a scan depend on scan type
and target behaviour — a filtered target that silently drops probes yields
`S0`, an actively refusing one yields `REJ`, etc. A production detector would
consider the broader set of scan-like states (`RSTRH`, `S0`, `REJ`, `RSTO`)
rather than `RSTRH` alone.